In [1]:
import pandas as pd
import os
import glob
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error,r2_score
import pickle

In [2]:
# Loading datas
df = pd.read_csv("metadata.csv")
df1 = pd.concat([pd.read_csv(f).assign(battery_id=os.path.basename(f).replace(".csv", ""))
                 for f in glob.glob("data/*.csv")], ignore_index=True)

In [3]:
df.head()

,type,start_time,ambient_temperature,battery_id,test_id,uid,filename,Capacity,Re,Rct
0,discharge,[2010. 7. 21. 15. 0. ...,4,B0047,0,1,00001.csv,1.6743047446975208,NaN,NaN
1,impedance,[2010. 7. 21. 16. 53. ...,24,B0047,1,2,00002.csv,NaN,0.05605783343888099,0.20097016584458333
2,charge,[2010. 7. 21. 17. 25. ...,4,B0047,2,3,00003.csv,NaN,NaN,NaN
3,impedance,[2010 7 21 20 31 5],24,B0047,3,4,00004.csv,NaN,0.05319185850921101,0.16473399914864734
4,discharge,[2.0100e+03 7.0000e+00 2.1000e+01 2.1000e+01 2...,4,B0047,4,5,00005.csv,1.5243662105099023,NaN,NaN


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7565 entries, 0 to 7564
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   type                 7565 non-null   object
 1   start_time           7565 non-null   object
 2   ambient_temperature  7565 non-null   int64 
 3   battery_id           7565 non-null   object
 4   test_id              7565 non-null   int64 
 5   uid                  7565 non-null   int64 
 6   filename             7565 non-null   object
 7   Capacity             2794 non-null   object
 8   Re                   1956 non-null   object
 9   Rct                  1956 non-null   object
dtypes: int64(3), object(7)
memory usage: 591.1+ KB


In [5]:
df.isnull().sum()

type                      0
start_time                0
ambient_temperature       0
battery_id                0
test_id                   0
uid                       0
filename                  0
Capacity               4771
Re                     5609
Rct                    5609
dtype: int64

In [6]:
# at metadata(df),there is useless columns,missing datas, and wrong dtypes .

In [7]:
# df data cleaning
df = df.drop(columns=["Re", "Rct", "filename", "start_time"])
df = df.dropna(subset=["Capacity"])
df["Capacity"] = pd.to_numeric(df["Capacity"], errors="coerce")

In [8]:
df.isna().sum()

type                    0
ambient_temperature     0
battery_id              0
test_id                 0
uid                     0
Capacity               25
dtype: int64

In [9]:
df = df.dropna(subset=["Capacity"])

In [10]:
df.isna().sum()

type                   0
ambient_temperature    0
battery_id             0
test_id                0
uid                    0
Capacity               0
dtype: int64

In [11]:
# df 1
df1.head()

,Voltage_measured,Current_measured,Temperature_measured,Current_load,Voltage_load,Time,battery_id,Current_charge,Voltage_charge,Sense_current,Battery_current,Current_ratio,Battery_impedance,Rectified_Impedance
0,4.162903,-0.002615,4.660864,0.0004,0.000,0.000,06825,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,4.162967,-0.001897,4.768639,0.0004,4.179,9.391,06825,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3.766685,-2.012164,4.889597,1.9982,2.767,19.891,06825,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3.743035,-2.012103,5.026764,1.9980,2.754,29.594,06825,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,3.723634,-2.013303,5.191478,1.9980,2.736,39.297,06825,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [12]:
df1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7376344 entries, 0 to 7376343
Data columns (total 14 columns):
 #   Column                Dtype  
---  ------                -----  
 0   Voltage_measured      float64
 1   Current_measured      float64
 2   Temperature_measured  float64
 3   Current_load          float64
 4   Voltage_load          float64
 5   Time                  float64
 6   battery_id            object 
 7   Current_charge        float64
 8   Voltage_charge        float64
 9   Sense_current         object 
 10  Battery_current       object 
 11  Current_ratio         object 
 12  Battery_impedance     object 
 13  Rectified_Impedance   object 
dtypes: float64(8), object(6)
memory usage: 787.9+ MB


In [13]:
df1.isnull().sum()

Voltage_measured          94145
Current_measured          94145
Temperature_measured      94145
Current_load            6606764
Voltage_load            6606764
Time                      93888
battery_id                    0
Current_charge           863468
Voltage_charge           863468
Sense_current           7282456
Battery_current         7282456
Current_ratio           7282456
Battery_impedance       7282456
Rectified_Impedance     7300060
dtype: int64

In [14]:
#df1 cleaning
cols_to_drop = ["Current_load", "Voltage_load", "Current_charge", "Voltage_charge",
                "Sense_current", "Battery_current", "Current_ratio", "Battery_impedance",
                "Rectified_Impedance"]
df1 = df1.drop(columns=[c for c in cols_to_drop if c in df1.columns])
df1 = df1.dropna()

In [15]:
df1.isna().sum()

Voltage_measured        0
Current_measured        0
Temperature_measured    0
Time                    0
battery_id              0
dtype: int64

In [16]:
# We are going to take mean for all batteries

In [17]:
df1_grouped = df1.groupby("battery_id").mean().reset_index()

In [18]:
#merge
# we need to convert uid columns to like; 1 = "00001", if we do not convert this column, datasets  can not be merge.
df["uid"] = df["uid"].astype(str).str.zfill(5)

df_merged = df.merge(df1_grouped, left_on="uid", right_on="battery_id", how="inner")
print(df_merged.shape)
df_merged.head()

(2768, 11)


,type,ambient_temperature,battery_id_x,test_id,uid,Capacity,battery_id_y,Voltage_measured,Current_measured,Temperature_measured,Time
0,discharge,4,B0047,4,00005,1.524366,00005,3.476559,-0.983889,8.210715,2815.872543
1,discharge,4,B0047,6,00007,1.508076,00007,3.470767,-0.983889,7.954455,2786.419943
2,discharge,4,B0047,8,00009,1.483558,00009,3.467551,-0.978947,7.985865,2761.858847
3,discharge,4,B0047,10,00011,1.467139,00011,3.462839,-0.976262,8.009427,2739.481436
4,discharge,4,B0047,12,00013,1.448858,00013,3.459184,-0.971168,8.033627,2720.716044


In [19]:
df_merged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2768 entries, 0 to 2767
Data columns (total 11 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   type                  2768 non-null   object 
 1   ambient_temperature   2768 non-null   int64  
 2   battery_id_x          2768 non-null   object 
 3   test_id               2768 non-null   int64  
 4   uid                   2768 non-null   object 
 5   Capacity              2768 non-null   float64
 6   battery_id_y          2768 non-null   object 
 7   Voltage_measured      2768 non-null   float64
 8   Current_measured      2768 non-null   float64
 9   Temperature_measured  2768 non-null   float64
 10  Time                  2768 non-null   float64
dtypes: float64(5), int64(2), object(4)
memory usage: 238.0+ KB


In [20]:
df_merged.isnull().sum()

type                    0
ambient_temperature     0
battery_id_x            0
test_id                 0
uid                     0
Capacity                0
battery_id_y            0
Voltage_measured        0
Current_measured        0
Temperature_measured    0
Time                    0
dtype: int64

In [21]:
df_merged["uid"].value_counts

<bound method IndexOpsMixin.value_counts of 0       00005
1       00007
2       00009
3       00011
4       00013
        ...  
2763    07554
2764    07556
2765    07558
2766    07562
2767    07564
Name: uid, Length: 2768, dtype: object>

In [22]:
df_merged["battery_id_y"].value_counts

<bound method IndexOpsMixin.value_counts of 0       00005
1       00007
2       00009
3       00011
4       00013
        ...  
2763    07554
2764    07556
2765    07558
2766    07562
2767    07564
Name: battery_id_y, Length: 2768, dtype: object>

In [23]:
df_merged["battery_id_x"].value_counts

<bound method IndexOpsMixin.value_counts of 0       B0047
1       B0047
2       B0047
3       B0047
4       B0047
        ...  
2763    B0055
2764    B0055
2765    B0055
2766    B0055
2767    B0055
Name: battery_id_x, Length: 2768, dtype: object>

In [24]:
df_merged["type"].value_counts

<bound method IndexOpsMixin.value_counts of 0       discharge
1       discharge
2       discharge
3       discharge
4       discharge
          ...    
2763    discharge
2764    discharge
2765    discharge
2766    discharge
2767    discharge
Name: type, Length: 2768, dtype: object>

In [25]:
# we do not need battery id y ,test id and uid

In [26]:
df_merged = df_merged.drop(columns=["battery_id_y", "uid", "test_id"])

In [27]:
# we need to save TableU before Encoding

In [28]:
df_tableau = df_merged.copy()

In [29]:
# encoding for type and battery_id_x

In [30]:
df_merged = pd.get_dummies(df_merged, columns=["type", "battery_id_x"])

In [31]:
df_merged.shape

(2768, 41)

In [32]:
df_merged.head()

,ambient_temperature,Capacity,Voltage_measured,Current_measured,Temperature_measured,Time,type_discharge,battery_id_x_B0005,battery_id_x_B0006,battery_id_x_B0007,...,battery_id_x_B0047,battery_id_x_B0048,battery_id_x_B0049,battery_id_x_B0050,battery_id_x_B0051,battery_id_x_B0052,battery_id_x_B0053,battery_id_x_B0054,battery_id_x_B0055,battery_id_x_B0056
0,4,1.524366,3.476559,-0.983889,8.210715,2815.872543,True,False,False,False,...,True,False,False,False,False,False,False,False,False,False
1,4,1.508076,3.470767,-0.983889,7.954455,2786.419943,True,False,False,False,...,True,False,False,False,False,False,False,False,False,False
2,4,1.483558,3.467551,-0.978947,7.985865,2761.858847,True,False,False,False,...,True,False,False,False,False,False,False,False,False,False
3,4,1.467139,3.462839,-0.976262,8.009427,2739.481436,True,False,False,False,...,True,False,False,False,False,False,False,False,False,False
4,4,1.448858,3.459184,-0.971168,8.033627,2720.716044,True,False,False,False,...,True,False,False,False,False,False,False,False,False,False


In [33]:
# features and target
X = df_merged.drop("Capacity", axis=1)
y = df_merged["Capacity"]

In [34]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.25,random_state=42)

In [35]:
print("X_train.shape:\n",X_train.shape)
print("X_test.shape:\n",X_test.shape)

X_train.shape:
 (2076, 40)
X_test.shape:
 (692, 40)


In [36]:
#Model

In [37]:
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train,y_train)

RandomForestRegressor(random_state=42)

In [38]:
#prediction
y_pred = model.predict(X_test)

In [39]:
mse = mean_squared_error(y_test,y_pred)
mae = mean_absolute_error(y_test,y_pred)
r2 = r2_score(y_test,y_pred)
print("mean squared error:",mse)
print()
print("Mean absolute error:",mae)
print()
print("Score:",r2)

mean squared error: 0.001518547395182449

Mean absolute error: 0.01436748986805714

Score: 0.9933298960908373


In [40]:
# ADDING PREDICTION
df_tableau["Predicted_Capacity"] = model.predict(df_merged.drop(columns=["Capacity"]))
df_tableau.to_csv("battery_tableau.csv", index=False)

In [41]:
# Saving model
with open("battery_model.pkl", "wb") as f:
    pickle.dump(model, f)

print("Saved model")

Saved model


In [42]:
print(df_tableau["Capacity"].dtype)
print(df_tableau["Capacity"].head())

float64
0    1.524366
1    1.508076
2    1.483558
3    1.467139
4    1.448858
Name: Capacity, dtype: float64
